# Phase 6: Advanced Models + Hyperparameter Tuning
XGBoost + LightGBM with Optuna TPE (Bayesian) optimization

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import joblib

try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False

from retail_iq.config import RAW_DATA_DIR, MODEL_DIR
from retail_iq.preprocessing import load_raw_data, preprocess_dates, merge_datasets
from retail_iq.features import FastFeatureEngineer
from retail_iq.evaluation import evaluate_model, create_metrics_table

## 1. Load and Prepare Data

In [ ]:
train, test, stores, oil, holidays, transactions = load_raw_data()
train, test, oil, holidays, transactions = preprocess_dates([train, test, oil, holidays, transactions])
df = merge_datasets(train, stores, oil, holidays, transactions)

CUTOFF = pd.Timestamp('2017-08-16')
train_df = df[df['date'] < CUTOFF].copy()
test_df = df[(df['date'] >= CUTOFF) & (df['date'] < '2017-09-01')].copy()

print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
def build_features(df, transactions, oil, stores, holidays):
    fe = FastFeatureEngineer(df, transactions=transactions, oil_price=oil, store_meta=stores, holidays=holidays)
    fe.add_temporal_features()
    fe.add_lag_and_rolling(lags=[1,7,14,365], windows=[7,14,28])
    fe.add_onpromotion_features()
    fe.add_macroeconomic_features()
    fe.add_transaction_features()
    fe.add_store_metadata()
    return fe.transform()

train_feat = build_features(train_df, transactions, oil, stores, holidays)
test_feat = build_features(test_df, transactions, oil, stores, holidays)

train_feat = train_feat.dropna(subset=['sales_lag_365d'])
test_feat = test_feat.dropna(subset=['sales_lag_365d'])

In [ ]:
exclude_cols = ['sales', 'date', 'family', 'city', 'state', 'holiday_type', 'locale', 'locale_name', 'description', 'transferred']
feature_cols = [c for c in train_feat.columns if c not in exclude_cols]

X_train = train_feat[feature_cols].fillna(0)
y_train = train_feat['sales'].values
X_test = test_feat[feature_cols].fillna(0)
y_test = test_feat['sales'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Features: {len(feature_cols)}')

## 2. XGBoost (Default)

In [ ]:
xgb_default = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42, n_jobs=-1)

xgb_default.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=50)

y_pred_xgb_default = xgb_default.predict(X_test_scaled)
metrics_xgb_default = evaluate_model(y_test, y_pred_xgb_default, 'XGBoost_Default')
print(metrics_xgb_default)

## 3. LightGBM (Default)

In [ ]:
lgb_default = lgb.LGBMRegressor(
    n_estimators=500, num_leaves=63, max_depth=7,
    learning_rate=0.05, min_child_samples=20, feature_fraction=0.8,
    objective='regression', random_state=42, n_jobs=-1, verbose=-1)

lgb_default.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])

y_pred_lgb_default = lgb_default.predict(X_test_scaled)
metrics_lgb_default = evaluate_model(y_test, y_pred_lgb_default, 'LightGBM_Default')
print(metrics_lgb_default)

## 4. Hyperparameter Tuning with Optuna

In [ ]:
if OPTUNA_AVAILABLE:
    def objective_xgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1}
        tscv = TimeSeriesSplit(n_splits=3)
        scores = []
        for train_idx, val_idx in tscv.split(X_train_scaled):
            X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, y_tr, verbose=False)
            y_pred = model.predict(X_val)
            scores.append(np.sqrt(np.mean((y_pred - y_val) ** 2)))
        return np.mean(scores)

    study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
    study_xgb.optimize(objective_xgb, n_trials=20, show_progress_bar=True)
    print(f'Best XGB params: {study_xgb.best_params}')
else:
    print('Optuna not available')

In [ ]:
if OPTUNA_AVAILABLE:
    def objective_lgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'objective': 'regression', 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
        tscv = TimeSeriesSplit(n_splits=3)
        scores = []
        for train_idx, val_idx in tscv.split(X_train_scaled):
            X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model = lgb.LGBMRegressor(**params)
            model.fit(X_tr, y_tr, callbacks=[lgb.early_stopping(50, verbose=False)])
            y_pred = model.predict(X_val)
            scores.append(np.sqrt(np.mean((y_pred - y_val) ** 2)))
        return np.mean(scores)

    study_lgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
    study_lgb.optimize(objective_lgb, n_trials=20, show_progress_bar=True)
    print(f'Best LGB params: {study_lgb.best_params}')
else:
    print('Optuna not available')

## 5. Train Tuned Models

In [ ]:
if OPTUNA_AVAILABLE and 'study_xgb' in locals():
    xgb_best_params = study_xgb.best_params
    xgb_best_params.update({'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1})
else:
    xgb_best_params = {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05,
                       'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'reg:squarederror',
                       'random_state': 42, 'n_jobs': -1}

xgb_tuned = xgb.XGBRegressor(**xgb_best_params)
xgb_tuned.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=50)

y_pred_xgb_tuned = xgb_tuned.predict(X_test_scaled)
metrics_xgb_tuned = evaluate_model(y_test, y_pred_xgb_tuned, 'XGBoost_Tuned')
print(metrics_xgb_tuned)

In [ ]:
if OPTUNA_AVAILABLE and 'study_lgb' in locals():
    lgb_best_params = study_lgb.best_params
    lgb_best_params.update({'objective': 'regression', 'random_state': 42, 'n_jobs': -1, 'verbose': -1})
else:
    lgb_best_params = {'n_estimators': 500, 'num_leaves': 63, 'max_depth': 7, 'learning_rate': 0.05,
                       'min_child_samples': 20, 'feature_fraction': 0.8, 'objective': 'regression',
                       'random_state': 42, 'n_jobs': -1, 'verbose': -1}

lgb_tuned = lgb.LGBMRegressor(**lgb_best_params)
lgb_tuned.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])

y_pred_lgb_tuned = lgb_tuned.predict(X_test_scaled)
metrics_lgb_tuned = evaluate_model(y_test, y_pred_lgb_tuned, 'LightGBM_Tuned')
print(metrics_lgb_tuned)

## 6. Save Models and Results

In [ ]:
date_str = datetime.now().strftime('%Y%m%d')

joblib.dump(xgb_tuned, f'{MODEL_DIR}/xgboost_tuned_{date_str}_v1.pkl')
joblib.dump(lgb_tuned, f'{MODEL_DIR}/lightgbm_tuned_{date_str}_v1.pkl')
joblib.dump(scaler, f'{MODEL_DIR}/standard_scaler.pkl')

with open(f'{MODEL_DIR}/best_params_xgboost.json', 'w') as f:
    json.dump(xgb_best_params, f, indent=2)
with open(f'{MODEL_DIR}/best_params_lightgbm.json', 'w') as f:
    json.dump(lgb_best_params, f, indent=2)

print('Models saved.')

In [ ]:
all_results = [metrics_xgb_default, metrics_lgb_default, metrics_xgb_tuned, metrics_lgb_tuned]
comparison_df = create_metrics_table(all_results)
print(comparison_df)
comparison_df.to_csv(f'{MODEL_DIR}/advanced_models_metrics.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x_pos = np.arange(len(comparison_df))

axes[0].bar(x_pos, comparison_df['RMSLE'])
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(comparison_df['model'], rotation=45, ha='right')
axes[0].set_ylabel('RMSLE')
axes[0].set_title('RMSLE by Model')

axes[1].bar(x_pos, comparison_df['R2'])
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(comparison_df['model'], rotation=45, ha='right')
axes[1].set_ylabel('R²')
axes[1].set_title('R² by Model')

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/advanced_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

- XGBoost and LightGBM trained with default and tuned parameters
- Optuna TPE used for Bayesian hyperparameter optimization
- TimeSeriesSplit(n_splits=3) for temporal CV

Next: Phase 7 - Evaluation + SHAP Analysis